# 13. Paoay: three-crop baseline with Unknown

**Primary view:** the saved carrot/cabbage/potato model with uncertain scores marked Unknown crop type. Missing inputs stay separate. No retraining or other-crop detector is introduced.

This crop-only view does not screen forest, built or other land cover. It is a conditional model result, not a complete land-cover map or verified crop area. The stricter seven-class/Dynamic World maps remain separate context products. See the [workflow guide](../../docs/paoay_workflow.md).

**Crop-map extent:** only the existing KMZ reference polygons within the study boundary. Outside is masked, not labelled Unknown. Notes remain unverified reference context. The separate Dynamic World map retains the study-wide context.


## 1. Setup and rules

Keep the current three-crop model fixed. Primary score rules remain probability ≥ 0.60 and margin ≥ 0.10; ties/weak scores become Unknown, not other crops. These are illustrative cutoffs, not calibrated correctness probabilities.

The optional context maps retain their existing Dynamic World rules unchanged. No fourth-class model is trained.


In [ ]:
from pathlib import Path
import json
import sys
repository_root = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                        if (folder / "pyproject.toml").is_file() and (folder / "src/crop_pipeline").is_dir()), None)
if repository_root is None:
    raise FileNotFoundError("Open this notebook from inside the project repository")
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))
from crop_pipeline.paoay import maps, crop_baseline
from crop_pipeline.analysis_workflow import resolve_stage_pointer, write_stage_pointer
from crop_pipeline.notebook_display import show_table
from crop_pipeline import progress as pipeline_progress
settings_path = repository_root / "settings.local.json"
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
def local_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (repository_root / path).resolve()
report_folder = local_path(local_settings.get("report_root", "./sen4agrinet_work/inventory"))
paoay_root = report_folder.parent / "paoay"
rules = {"minimum_crop_probability": 0.60, "minimum_crop_margin": 0.10, "minimum_supported_bins": 4}
show_preview = True
show_context_preview = False  # Optional conservative Dynamic World map, separate from the crop baseline.
prediction_pointer = paoay_root / "predictions/prediction_stage.json"
map_root = paoay_root / "maps"
map_pointer = map_root / "map_stage.json"
write_stage_pointer(map_pointer, status="pending", reason="Notebook 13 has not finished current map assembly")
stages = ["Verify prediction and guidance inputs", "Assemble attributed maps", "Review class areas and sources", "Preview and verify maps", "Summary and next action"]
notebook_progress = pipeline_progress.start_notebook("13_paoay_maps", stages, report_folder.parent / "progress", {"rules": rules})


## 2. Verify matching inputs

Follow notebook 12’s exact feature/guidance chain. Original grid, dates, model, probabilities and crop-only rasters remain unchanged.


In [ ]:
pipeline_progress.begin_stage("Verify prediction and guidance inputs")
prediction_handoff = resolve_stage_pointer(prediction_pointer)
request = maps.prepare_request(prediction_handoff, map_root, rules)
baseline_request = crop_baseline.prepare_request(prediction_handoff, paoay_root / "crop_baseline", {"minimum_top_probability": rules["minimum_crop_probability"], "minimum_margin": rules["minimum_crop_margin"]})
show_table([{"rule": key.replace("_", " "), "value": value} for key, value in rules.items()], title="Illustrative assembly rules")
show_table([{"code": key, "class": value} for key, value in maps.CLASSES.items()], title="Seven thesis classes")
show_table([{"code": key, "status": value} for key, value in maps.STATUSES.items()], title="Separate data statuses (not classes)")
pipeline_progress.end_stage()


## 3. Prepare separate products

Create the primary three-crop + Unknown view directly from saved model scores. It does not require Dynamic World agreement. Missing crop inputs remain missing.

Also retain the conservative context maps as a separate product, preserving their class/status/source bands. Their larger unknown area must not be confused with the crop model's own uncertainty.


In [ ]:
pipeline_progress.begin_stage("Assemble attributed maps")
def map_progress(items):
    return pipeline_progress.track(items, "Assemble map contexts", unit="half-month")
crop_baseline_handoff = crop_baseline.build(baseline_request)
baseline_bundle = crop_baseline.load_bundle(crop_baseline_handoff)
map_handoff = maps.build(request, iterator=map_progress)
bundle = maps.load_map_bundle(map_handoff)
pipeline_progress.end_stage()


## 4. Review the primary result

Carrot/cabbage/potato predictions plus Unknown and Missing input account for the reference-polygon union once. Raw probabilities, features and the fitted model remain unchanged. A confident three-crop model prediction can still be wrong on a non-crop surface.


In [ ]:
pipeline_progress.begin_stage("Review class areas and sources")
show_table([{**r, "pixels": int(r["pixels"]), "area_ha": float(r["area_ha"]), "roi_percent": float(r["roi_percent"])} for r in baseline_bundle["summary"]], title="PRIMARY: current three-crop model + Unknown")
print("Crop-only model scores; these areas are not verified planted area.")
print("Conservative Dynamic World context tables:", bundle["folder"])
print("Other-crop training methods: TODO only.")
pipeline_progress.end_stage()


## 5. Preview and publish

The default preview shows the three-crop model plus Unknown. Enable `show_context_preview` to inspect the separate seven-class/Dynamic World result. This is not independent local accuracy.


In [ ]:
pipeline_progress.begin_stage("Preview and verify maps")
if show_preview:
    from IPython.display import display
    import matplotlib.pyplot as plt
    figure = crop_baseline.plot(baseline_bundle)
    preview = paoay_root / "crop_baseline/previews" / (baseline_bundle["manifest"]["bundle_id"] + ".png")
    preview.parent.mkdir(parents=True, exist_ok=True)
    if not preview.exists():
        figure.savefig(preview, dpi=150)
    display(figure)
    plt.close(figure)
    print("Primary crop baseline preview:", preview)
if show_context_preview:
    from IPython.display import display
    import matplotlib.pyplot as plt
    context_figure = maps.plot_maps(bundle)
    display(context_figure)
    plt.close(context_figure)
maps.load_map_bundle(map_handoff)
write_stage_pointer(map_pointer, map_handoff)
pipeline_progress.end_stage()


## 6. Summary and next action

The final map series remains a fixed-calendar prototype with unverified crop-cycle alignment. It does not establish crop accuracy.


In [ ]:
pipeline_progress.begin_stage("Summary and next action")
print("Primary crop baseline handoff:", crop_baseline_handoff)
print("Separate context map handoff:", map_handoff)
print("Current map pointer:", map_pointer)
print("Raw crop model outputs and fitted model: preserved")
print("Next: notebook 14 reviews area summaries beside the source notes.")
print("Dynamic World attribution:", bundle["manifest"]["attribution"])
print("Detail log:", notebook_progress.output.path)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()


## TODO — alternative training methods (deferred)

- [ ] Train a separate four-class model pooling selected labelled non-target crops.
- [ ] Train individual crop labels and group known non-target predictions into “other crops” for reporting.

Preserve parcel splits, balance crop/source sampling, and test crop species withheld from training. Neither method is implemented here. Low confidence remains Unknown.
